In [ ]:
!pip install geopandas folium earthengine-api geemap -q

In [ ]:
import json
import folium
import geopandas as gpd
import ee

ee.Authenticate()
ee.Initialize(project="siberian-487118")

In [ ]:
gdf = gpd.read_file('khmao.geojson')

with open('khmao.geojson') as f:
    geojson_data = json.load(f)

features = geojson_data['features']
aoi = ee.Geometry(features[0]['geometry'])

In [ ]:
import json
from io import BytesIO

import ee
import matplotlib.pyplot as plt
import requests
from PIL import Image


collage_bbox = ee.Geometry.BBox(59.8, 60.8, 70, 65)


cities = [
    {
        "name": "Agirish",
        "coords": [61.91865539550781, 63.02293014526367],
        "ha": "left",
        "va": "bottom",
    },
    {
        "name": "Yugorsk",
        "coords": [61.3149709, 63.3315999],
        "ha": "right",
        "va": "bottom",
    },
    {
        "name": "Nyagan",
        "coords": [63.191513, 64.424301],
        "ha": "left",
        "va": "top",
    },
    {
        "name": "Igrim",
        "coords": [65.43, 62.14],
        "ha": "left",
        "va": "top",
    },
    {
        "name": "Khanty-Mansiysk",
        "coords": [69.02, 61.00],
        "ha": "left",
        "va": "bottom",
    },
]

city_features = ee.FeatureCollection(
    [
        ee.Feature(ee.Geometry.Point(c["coords"]), {"name": c["name"]})
        for c in cities
    ]
)


def add_modis_ndvi(image):
    ndvi = image.normalizedDifference(
        ["sur_refl_b02", "sur_refl_b01"]
    ).rename("NDVI")
    return image.addBands(ndvi)


def get_modis_mosaic(year, geometry):
    start_date = f"{year}-08-01"
    end_date = f"{year}-09-15"

    collection = (
        ee.ImageCollection("MODIS/061/MOD09A1")
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .map(add_modis_ndvi)
    )

    return collection.qualityMosaic("NDVI").clip(geometry)


modis_vis = {
    "bands": ["sur_refl_b07", "sur_refl_b02", "sur_refl_b01"],
    "min": 0,
    "max": 3000,
}

target_years = [2020, 2021, 2022]
images_dict = {}

print("Fetching 3 cloud-free MODIS frames with borders from Earth Engine...")

for year in target_years:
    mosaic = get_modis_mosaic(year, collage_bbox)
    visualized_base = mosaic.visualize(**modis_vis)
    
    khmao_outline = ee.Image().paint(aoi, 'color', 5)
    visualized_with_border = visualized_base.blend(khmao_outline.visualize(palette=['#FFFFFF']))
    city_dots = city_features.draw(color='#FF3333', pointRadius=4)
    final_visualized = visualized_with_border.blend(city_dots)
    
    flat_visualized = final_visualized.reproject(crs='EPSG:3857', scale=500)
    
    thumb_url = flat_visualized.getThumbURL({
        'region': collage_bbox,
        'dimensions': 1200,
        'format': 'png'  
    })
    
    response = requests.get(thumb_url)
    img = Image.open(BytesIO(response.content))
    images_dict[year] = img

lon_min, lat_min, lon_max, lat_max = 59.8, 60.8, 70.0, 65.0
import numpy as np
mean_lat = (lat_min + lat_max) / 2
geo_aspect = 1.0 / np.cos(np.radians(mean_lat))

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
axes_flat = axes.flatten()

for i, year in enumerate(target_years):
    axes_flat[i].imshow(images_dict[year], extent=[lon_min, lon_max, lat_min, lat_max], aspect=geo_aspect)
    axes_flat[i].set_title(f'August/September {year} (MODIS 500m)', fontsize=16, fontweight='semibold')
    
    for city in cities:
        if lon_min <= city["coords"][0] <= lon_max and lat_min <= city["coords"][1] <= lat_max:
            axes_flat[i].text(
                city["coords"][0], city["coords"][1], f' {city["name"]}', 
                color='#FFFF00', fontsize=11, fontweight='bold',
                horizontalalignment=city["ha"], verticalalignment=city["va"],
                bbox=dict(facecolor='black', alpha=0.4, boxstyle='round,pad=0.2', edgecolor='none')
            )
            
    axes_flat[i].set_xlabel("Longitude", fontsize=12)
    if i == 0:
        axes_flat[i].set_ylabel("Latitude", fontsize=12)
    else:
        axes_flat[i].get_yaxis().set_visible(False) 

plt.tight_layout()
plt.show()